# Implementação de Sistema de Raciocínio Baseado em Casos (RBC)
## Domínio: Avaliação de Preço de Imóveis

Implementa do ciclo completo de RBC (Recuperar, Reusar, Revisar, Reter)

**Aluno:** Arthur de Barros Marcondes Machado Rosisca
**Universidade:** UTFPR-PG

## Configuração: Imports e Constantes

Nesta célula, importamos as bibliotecas (Pandas, Math) e definimos as constantes globais do nosso sistema:

1.  **`NOME_ARQUIVO_BASE_CASOS`**: O arquivo CSV que armazena nossa Base de Casos (Case Library).
2.  **`PESOS`**: Define a importância (peso) de cada atributo na hora de calcular a similaridade. A soma total deve ser 1.0.
3.  **`TABELA_SIMILARIDADE_BAIRRO`**: Implementa a **Similaridade Local** para atributos simbólicos não ordenados. Definimos manualmente o quão similar um bairro é do outro (ex: Centro é 70% similar ao Jardim Carvalho, mas só 20% similar a Uvaranas).

In [1]:
import math
import pandas as pd

NOME_ARQUIVO_BASE_CASOS = "base_de_casos.csv"

PESOS = {
    "metros_quadrados": 0.30,  
    "bairro": 0.25,
    "num_quartos": 0.20,      
    "num_banheiros": 0.10,
    "vagas_garagem": 0.10,
    "mobiliado": 0.0          
}

soma_pesos = sum(PESOS.values())
if not math.isclose(soma_pesos, 1.0):
    print(f"Atenção: A soma dos pesos é {soma_pesos}, mas deveria ser 1.0. Ajustando...")
    for chave in PESOS:
        PESOS[chave] /= soma_pesos

TABELA_SIMILARIDADE_BAIRRO = {
    "Centro": {
        "Centro": 1.0,
        "Jardim Carvalho": 0.7, 
        "Oficinas": 0.3, 
        "Uvaranas": 0.2
    },
    "Jardim Carvalho": {
        "Centro": 0.7, 
        "Jardim Carvalho": 1.0, 
        "Oficinas": 0.4, 
        "Uvaranas": 0.3
    },
    "Oficinas": {
        "Centro": 0.3, 
        "Jardim Carvalho": 0.4, 
        "Oficinas": 1.0, 
        "Uvaranas": 0.8
    },
    "Uvaranas": {
        "Centro": 0.2, 
        "Jardim Carvalho": 0.3, 
        "Oficinas": 0.8, 
        "Uvaranas": 1.0
    }
}

print("Imports e Constantes carregados (com Pandas e CSV).")

Atenção: A soma dos pesos é 0.95, mas deveria ser 1.0. Ajustando...
Imports e Constantes carregados (com Pandas e CSV).


## Carregando a Base de Casos e Pré-processamento

Aqui, definimos duas funções cruciais:

1.  **`carregar_base_de_casos`**: Lê o arquivo `base_de_casos.csv`.
2.  **`calcular_ranges_normalizacao`**: Esta função é essencial. Ela encontra os valores `min` e `max` de cada atributo numérico na base. Isso é necessário para a **Normalização**. "Atributos com escala mais ampla/maior variabilidade tendem a ter maior peso nos cálculos de distâncias". 

Para evitar isso, usaremos a "Reescala Linear [0,1]" (ou Normalização Min-Max) em todos os cálculos de similaridade numérica.

In [2]:
def carregar_base_de_casos(arquivo_csv):
    try:
        df = pd.read_csv(arquivo_csv, index_col=0) 
        
        df.index.name = 'id_caso'
        
        print(f"Base de casos (CSV) carregada em um DataFrame. {len(df)} casos encontrados.")
        return df
    except FileNotFoundError:
        print(f"Erro: Arquivo '{arquivo_csv}' não encontrado. Execute a célula de conversão primeiro.")
        return pd.DataFrame() 

def calcular_ranges_normalizacao(df):
    ranges = {}
    atributos_numericos = ["metros_quadrados", "num_quartos", "num_banheiros", "vagas_garagem"]
    
    if df.empty:
        return None

    for atributo in atributos_numericos:
        ranges[atributo] = {
            "min": df[atributo].min(),
            "max": df[atributo].max()
        }
    
    print("Valores Min-Max para normalização calculados:")
    
    ranges_df = pd.DataFrame(ranges).transpose() 
    display(ranges_df)
    
    return ranges 

df_base_de_casos = carregar_base_de_casos(NOME_ARQUIVO_BASE_CASOS)
ranges_normalizacao = calcular_ranges_normalizacao(df_base_de_casos)

if not df_base_de_casos.empty:
    display(df_base_de_casos.head())

Base de casos (CSV) carregada em um DataFrame. 52 casos encontrados.
Valores Min-Max para normalização calculados:


,min,max
metros_quadrados,42.0,160.0
num_quartos,1.0,4.0
num_banheiros,1.0,3.0
vagas_garagem,0.0,3.0


,metros_quadrados,num_quartos,num_banheiros,vagas_garagem,bairro,mobiliado,preco_vendido
id_caso,,,,,,,
1,65.0,2,1,1,Centro,True,395812.25
2,80.0,2,2,1,Centro,False,451555.20
3,50.0,1,1,0,Uvaranas,False,194875.00
4,120.0,4,2,2,Jardim Carvalho,True,881408.40
5,45.0,1,1,1,Oficinas,True,230723.62


## Calculo do custo medio de mobilia

In [3]:
def calcular_custos_por_bairro(df):
    """
    Calcula o "valor" da mobília por m² para CADA BAIRRO.
    """
    
    custos_mobilia_por_bairro = {}
    bairros = df['bairro'].unique()
    
    df_temp = df.copy()
    df_temp['preco_por_mq'] = df_temp['preco_vendido'] / df_temp['metros_quadrados']
    
    print(f"Pré-cálculo de adaptação concluído:")
    
    for bairro in bairros:
        casos_bairro = df_temp[df_temp['bairro'] == bairro]
        
        media_mobiliado = casos_bairro[casos_bairro['mobiliado'] == True]['preco_por_mq'].mean()
        media_nao_mobiliado = casos_bairro[casos_bairro['mobiliado'] == False]['preco_por_mq'].mean()
        
        if pd.isna(media_mobiliado) or pd.isna(media_nao_mobiliado):
            
            custo_mobilia_por_mq = 0.0
            print(f"  -> {bairro}: Faltam dados para calcular (usando R$ 0.00)")
        else:
            custo_mobilia_por_mq = media_mobiliado - media_nao_mobiliado
            print(f"  -> {bairro}: Custo da Mobília = R$ {custo_mobilia_por_mq:,.2f} por m²")
            
        custos_mobilia_por_bairro[bairro] = custo_mobilia_por_mq
        
    return custos_mobilia_por_bairro

# --- Executa o novo cálculo ---
custos_mobilia_por_bairro = calcular_custos_por_bairro(df_base_de_casos)

Pré-cálculo de adaptação concluído:
  -> Centro: Custo da Mobília = R$ 849.98 por m²
  -> Uvaranas: Custo da Mobília = R$ 889.07 por m²
  -> Jardim Carvalho: Custo da Mobília = R$ 1,141.17 por m²
  -> Oficinas: Custo da Mobília = R$ 451.85 por m²


## Etapa 1: RECUPERAÇÃO (Retrieve)

Esta é a primeira etapa do ciclo RBC. O objetivo é encontrar na base de casos o problema anterior mais similar ao novo problema.

Para isso, implementamos as seguintes funções:

1.  **`normalizar_valor`**: Aplica a fórmula de Reescala Linear [0,1] (`(valor - min) / (max - min)`).
2.  **`calcular_similaridade_local`**: Calcula a similaridade para um *único* atributo.
    * Para **Números** (ex: `metros_quadrados`): `1 - abs(valor_novo_norm - valor_caso_norm)`.
    * Para **Símbolos** (ex: `bairro`, `mobiliado`): Usa a tabela de similaridade ou a correspondência exata.
3.  **`calcular_similaridade_global`**: Combina todas as similaridades locais usando os `PESOS` que definimos.
4.  **`recuperar_caso_mais_similar`**: Implementa a **Recuperação Sequencial**. Ela aplica a `calcular_similaridade_global` a todos os casos da base e retorna o "vizinho mais próximo" (Nearest Neighbour).

In [4]:
def normalizar_valor(valor, ranges_atributo):
    
    min_val = ranges_atributo["min"]
    max_val = ranges_atributo["max"]

    if max_val == min_val:
        return 0
    
    return (valor - min_val) / (max_val - min_val)

def calcular_similaridade_local(valor_novo, valor_caso, atributo, ranges_normalizacao):
    
    if atributo in ranges_normalizacao:
        ranges_attr = ranges_normalizacao[atributo]
        
        v_novo_norm = normalizar_valor(valor_novo, ranges_attr)
        v_caso_norm = normalizar_valor(valor_caso, ranges_attr)
        
        return 1 - abs(v_novo_norm - v_caso_norm)
    
    elif atributo == "bairro":
        try:
            return TABELA_SIMILARIDADE_BAIRRO[valor_novo][valor_caso]
        except KeyError:
            return 0.0
            
    elif atributo == "mobiliado":
        return 1.0 if valor_novo == valor_caso else 0.0
    
    else:
        return 0.0

def calcular_similaridade_global(linha_caso, problema_novo, pesos, ranges_normalizacao):
    
    similaridade_total = 0.0
    
    for atributo, peso in pesos.items():
        if atributo in problema_novo and atributo in linha_caso:
            sim_local = calcular_similaridade_local(
                problema_novo[atributo],
                linha_caso[atributo],
                atributo,
                ranges_normalizacao
            )
            similaridade_total += sim_local * peso
            
    return similaridade_total

def recuperar_caso_mais_similar(df, problema_novo, pesos, ranges_normalizacao):
    """
    Implementa a Recuperação usando Pandas .apply().
    """
    if df.empty:
        return None, 0.0

    df_temp = df.copy()
    
    df_temp['similaridade'] = df_temp.apply(
        calcular_similaridade_global,
        axis=1,
        problema_novo=problema_novo,
        pesos=pesos,
        ranges_normalizacao=ranges_normalizacao
    )
    
    id_melhor_caso = df_temp['similaridade'].idxmax()
    
    max_similaridade = df_temp['similaridade'].max()
    
    melhor_caso_dict = df_temp.loc[id_melhor_caso].to_dict()
    
    melhor_caso_dict['id_caso'] = id_melhor_caso
            
    return melhor_caso_dict, max_similaridade, df_temp

print("Funções da Etapa 1 (Recuperação) definidas.")

Funções da Etapa 1 (Recuperação) definidas.


## Função Auxiliar: Obter Entrada do Usuário

Esta célula define uma função simples (`obter_novo_problema`) para coletar os dados do **Novo Problema** (ou `New Case`) do usuário.

In [5]:
def obter_novo_problema():
    
    print("\n--- Por favor, insira os dados do novo imóvel ---")
    
    try:
        problema = {}
        problema["metros_quadrados"] = float(input("Metros quadrados (ex: 70): "))
        problema["num_quartos"] = int(input("Número de quartos (ex: 3): "))
        problema["num_banheiros"] = int(input("Número de banheiros (ex: 1): "))
        problema["vagas_garagem"] = int(input("Vagas de garagem (ex: 1): "))
        
        bairros_validos = list(TABELA_SIMILARIDADE_BAIRRO.keys())
        while True:
            bairro = input(f"Bairro {bairros_validos}: ")
            if bairro in bairros_validos:
                problema["bairro"] = bairro
                break
            else:
                print(f"Bairro inválido. Por favor, escolha um dos seguintes: {bairros_validos}")
        
        while True:
            mobiliado_input = input("É mobiliado? (s/n): ").lower()
            if mobiliado_input == 's':
                problema["mobiliado"] = True
                break
            elif mobiliado_input == 'n':
                problema["mobiliado"] = False
                break
            else:
                print("Entrada inválida. Digite 's' para sim ou 'n' para não.")
                
        return problema
    
    except ValueError:
        print("Erro: Entrada inválida. Por favor, insira números corretos.")
        return None

print("Função 'obter_novo_problema' definida.")

Função 'obter_novo_problema' definida.


## Etapa 2: REUSO (Reuse)

Esta é a segunda e mais crucial etapa do ciclo, onde a **previsão de preço** é de fato realizada.

O objetivo é pegar a solução do caso recuperado (o `preco_vendido` antigo) e **adaptá-la** para o novo problema. O requisito do trabalho é claro: "Implementar uma forma de **adaptação automática (exceto a adaptação nula)**".

Para isso, implementamos uma **Adaptação Transformacional Substitucional**, que usa "conhecimento do domínio... na forma de regras ou fórmulas".

Nossa função `reusar_solucao_adaptada` (V4) faz a previsão em dois passos:

1.  **Cálculo Base (Preço/m²):** A função calcula o `preço/m²` do caso antigo recuperado. Este valor é a **base da nossa previsão**.
2.  **Adaptação Principal:** Esse `preço/m²` é então aplicado aos `metros_quadrados` do novo problema, gerando o `preco_base_adaptado`.
3.  **Regra de Ajuste (Mobília):** O sistema aplica um ajuste contextual. Ele usa o valor de mobília (em R$/m²) que ele *aprendeu* da base de dados (na Célula 6.5) para adicionar ou remover valor, caso o status de `mobiliado` seja diferente entre o novo problema e o caso antigo.

In [6]:
def reusar_solucao_adaptada(caso_recuperado, novo_problema, custos_mobilia_por_bairro):
    """
    Adapta a solução usando o custo da mobília contextual (por bairro).
    """
    
    preco_antigo = caso_recuperado['preco_vendido']
    mq_antigo = caso_recuperado['metros_quadrados']
    mobiliado_antigo = caso_recuperado['mobiliado']
    bairro_antigo = caso_recuperado['bairro'] 
    
    mq_novo = novo_problema['metros_quadrados']
    mobiliado_novo = novo_problema['mobiliado']
    
    if mq_antigo == 0:
        return preco_antigo 

    # 3. Calcula o preço/m² do caso antigo
    preco_por_mq_antigo = preco_antigo / mq_antigo
    
    # 4. Adaptação base (Substituição): Calcula o novo preço com base no novo m²
    preco_base_adaptado = preco_por_mq_antigo * mq_novo
    
    print(f"Preço Base (por m²): R$ {preco_base_adaptado:,.2f}")
    
    # 5. Regra de Adaptação V4: Ajusta o preço com base na mobília e bairro
    
    # Pega o custo da mobília específico para o bairro do caso recuperado
    custo_mobilia_por_mq = custos_mobilia_por_bairro.get(bairro_antigo, 0.0)
    
    # Calcula o valor do ajuste EM REAIS (R$)
    valor_ajuste_mobilia = custo_mobilia_por_mq * mq_novo
    
    
    if mobiliado_novo and not mobiliado_antigo:
        # O novo é mobiliado e o antigo não era -> ADICIONA o valor da mobília
        preco_final_adaptado = preco_base_adaptado + valor_ajuste_mobilia
        print(f"INFO: Aplicado ajuste de +R$ {valor_ajuste_mobilia:,.2f} (Valor da Mobília para '{bairro_antigo}')")
        
    elif not mobiliado_novo and mobiliado_antigo:
        # O novo NÃO é mobiliado e o antigo era -> REMOVE o valor da mobília
        preco_final_adaptado = preco_base_adaptado - valor_ajuste_mobilia
        print(f"INFO: Aplicado ajuste de -R$ {valor_ajuste_mobilia:,.2f} (Remoção da Mobília para '{bairro_antigo}')")
        
    else:
        # Ambos são iguais (mobiliados ou não-mobiliados), preço base se mantém
        preco_final_adaptado = preco_base_adaptado
        print("INFO: Nenhum ajuste de mobília necessário (status de mobília é o mesmo).")

    # Retorna APENAS o preço final, a etapa de revisão usará este valor.
    return preco_final_adaptado

print("Funções da Etapa 2 (Reuso) ATUALIZADAS (V4.1 - Com print do preço base).")

Funções da Etapa 2 (Reuso) ATUALIZADAS (V4.1 - Com print do preço base).


## Etapa 3: REVISÃO (Revise)

Esta é a terceira etapa do ciclo. O objetivo da Revisão é "avaliar a solução gerada pelo reuso (adaptação)" e, se necessário, "reparar a solução para o caso".

A aula menciona que a avaliação pode ser feita por "Interação com usuário".

A implementação (`revisar_solucao`) fará exatamente isso:
1.  Apresentar a `solucao_sugerida` ao usuário.
2.  Perguntar se a solução foi correta.
3.  Se a solução foi incorreta (ex: o imóvel foi vendido por um valor diferente), vamos "reparar" pedindo ao usuário que digite o valor correto.
4.  O resultado desta etapa é um "Caso reparado/testado", que estará pronto para a próxima etapa (Retenção).

In [7]:
def revisar_solucao(solucao_sugerida):
    
    print(f"\nO sistema sugeriu um preço de R$ {solucao_sugerida:,.2f}.")
    
    while True:
        try:
            feedback = input("Essa sugestão foi precisa? (s/n): ").lower()
            
            if feedback == 's':
                # ação Correta -> continuar com a retenção
                print("INFO: Solução confirmada pelo usuário.")
                return solucao_sugerida
                
            elif feedback == 'n':
                # ação Incorreta -> reparar a solução
                print("INFO: Solução marcada como incorreta. Iniciando reparo...")
                
                # aede ao usuário a solução correta
                valor_correto = float(input("Por favor, digite o valor final (correto) de venda (ex: 525000): "))
                print(f"INFO: Solução reparada para R$ {valor_correto:,.2f}.")
                
                return valor_correto
                
            else:
                print("Entrada inválida. Digite 's' para Sim ou 'n' para Não.")
                
        except ValueError:
            print("Erro: Valor inválido. Por favor, digite um número.")
        except Exception as e:
            print(f"Um erro ocorreu: {e}")
            return None # retorna none em caso de erro

print("Funções da Etapa 3 (Revisão) definidas.")

Funções da Etapa 3 (Revisão) definidas.


## Etapa 4: RETENÇÃO (Retain)

Esta é a quarta e última etapa do ciclo RBC. O objetivo da Retenção é o "processo de incorporação, ao conhecimento já existente, daquilo que é útil de um novo episódio de solução de um problema".

Isso torna o sistema um "solucionador de problemas mais poderoso com o passar do tempo e seu uso".

Nossa implementação (`reter_novo_caso`) fará o seguinte:
1.  Pegar o `novo_problema` e a `solucao_confirmada` (da Etapa 3).
2.  Formatar esses dados como um novo caso.
3.  Fazer a "Integração na base de casos" através da "Adição... de casos".
4.  Em termos de Pandas, vamos adicionar uma nova linha ao nosso DataFrame e salvar o arquivo CSV de volta ao disco.

In [8]:
def reter_novo_caso(df_base, problema_novo, solucao_confirmada):
    
    try:
        # 1. encontra o próximo ID disponível
        id_novo_caso = df_base.index.max() + 1
        
        # 2. cria o dicionário do novo caso
        novo_caso_dict = {}
        novo_caso_dict.update(problema_novo) # Adiciona os atributos do problema
        novo_caso_dict['preco_vendido'] = solucao_confirmada # Adiciona a solução
        
        # 3. converte o dicionário para um DataFrame de 1 linha
        # o 'index=[id_novo_caso]' garante que ele tenha o novo ID
        novo_caso_df = pd.DataFrame(novo_caso_dict, index=[id_novo_caso])
        
        # 4. concatena o novo caso ao DataFrame principal
        df_atualizado = pd.concat([df_base, novo_caso_df])
        
        # 5. Salva o DataFrame atualizado de volta no CSV
        df_atualizado.to_csv('base_de_casos.csv')
        
        print(f"\n--- ETAPA 4: RETENÇÃO (Retain) ---")
        print(f"Sucesso! Novo caso (ID: {id_novo_caso}) foi aprendido e salvo na base.")
        
        return df_atualizado
        
    except Exception as e:
        print(f"ERRO na retenção: Não foi possível salvar o novo caso. {e}")
        return df_base # Retorna o DataFrame original sem salvar

print("Funções da Etapa 4 (Retenção) definidas.")

Funções da Etapa 4 (Retenção) definidas.


# CICLO RBC COMPLETO (Etapas 1-4) - Demonstração Final

Agora, vamos integrar todas as 4 etapas em um único fluxo de execução. Esta célula irá:
1.  **Obter** o novo problema.
2.  **Recuperar** o caso mais similar.
3.  **Reusar** (adaptar) a solução antiga para criar uma sugestão.
4.  **Revisar** a sugestão com o usuário e obter uma solução final.
5.  **Reter** o novo problema e sua solução final na base de casos.

Ao final, o sistema recalcula os ranges de normalização, mostrando que "aprendeu" e está pronto para a próxima consulta.

In [11]:
if 'df_base_de_casos' not in globals() or df_base_de_casos.empty:
    print("Recarregando base de casos...")
    df_base_de_casos = carregar_base_de_casos(NOME_ARQUIVO_BASE_CASOS)
    ranges_normalizacao = calcular_ranges_normalizacao(df_base_de_casos)
    # Recalcula o custo da mobília (por bairro) ao carregar
    custos_mobilia_por_bairro = calcular_custos_por_bairro(df_base_de_casos)

if not df_base_de_casos.empty:
    
    novo_problema = obter_novo_problema()
    
    if novo_problema:
        # --- ETAPA 1: RECUPERAÇÃO ---
        print("\n--- ETAPA 1: RECUPERAÇÃO (Retrieve) ---")
        caso_recuperado, similaridade, df_com_similaridade = recuperar_caso_mais_similar(
            df_base_de_casos,
            novo_problema,
            PESOS,
            ranges_normalizacao
        )
        
        if not caso_recuperado:
            print("Erro: Nenhum caso recuperado. Abortando.")
        else:
            print(f"Caso mais similar encontrado (ID: {caso_recuperado['id_caso']}) com {similaridade*100:.2f}% de similaridade.")
            print(f"  -> Solução Antiga: R$ {caso_recuperado['preco_vendido']:,.2f}")
            display(df_com_similaridade.sort_values(by='similaridade', ascending=False).head(3))
            
            # --- ETAPA 2: REUSO ---
            print("\n--- ETAPA 2: REUSO (Reuse/Adapt) ---")
            # PASSAMOS O NOVO DICIONÁRIO AQUI
            solucao_sugerida = reusar_solucao_adaptada(caso_recuperado, novo_problema, custos_mobilia_por_bairro)
            
            # --- AJUSTE NO PRINT ABAIXO ---
            print(f"\nPreço Final Sugerido (Após Ajustes): R$ {solucao_sugerida:,.2f}")
            # --- FIM DO AJUSTE ---
            
            # --- ETAPA 3: REVISÃO ---
            print("\n--- ETAPA 3: REVISÃO (Revise) ---")
            solucao_confirmada = revisar_solucao(solucao_sugerida)
            
            # --- ETAPA 4: RETENÇÃO ---
            if solucao_confirmada is not None:
                df_base_de_casos = reter_novo_caso(df_base_de_casos, novo_problema, solucao_confirmada)
                
                print("INFO: Recalculando parâmetros do sistema com o novo caso...")
                ranges_normalizacao = calcular_ranges_normalizacao(df_base_de_casos)
                # Recalcula o custo da mobília (por bairro)
                custos_mobilia_por_bairro = calcular_custos_por_bairro(df_base_de_casos)
                
                print("\n--- Ciclo RBC Concluído ---")
                print(f"A base de casos agora possui {len(df_base_de_casos)} casos.")
                display(df_base_de_casos.tail(3)) 
            
            else:
                print("Revisão cancelada. O novo caso não será retido.")

else:
    print("O programa não pode iniciar sem uma base de casos.")


--- Por favor, insira os dados do novo imóvel ---


Metros quadrados (ex: 70):  80
Número de quartos (ex: 3):  2
Número de banheiros (ex: 1):  1
Vagas de garagem (ex: 1):  1
Bairro ['Centro', 'Jardim Carvalho', 'Oficinas', 'Uvaranas']:  Centro
É mobiliado? (s/n):  s



--- ETAPA 1: RECUPERAÇÃO (Retrieve) ---
Caso mais similar encontrado (ID: 36) com 98.13% de similaridade.
  -> Solução Antiga: R$ 401,500.00


,metros_quadrados,num_quartos,num_banheiros,vagas_garagem,bairro,mobiliado,preco_vendido,similaridade
36,73.0,2,1,1,Centro,False,401500.00,0.981267
1,65.0,2,1,1,Centro,True,395812.25,0.959857
2,80.0,2,2,1,Centro,False,451555.20,0.947368



--- ETAPA 2: REUSO (Reuse/Adapt) ---
Preço Base (por m²): R$ 440,000.00
INFO: Aplicado ajuste de +R$ 67,998.69 (Valor da Mobília para 'Centro')

Preço Final Sugerido (Após Ajustes): R$ 507,998.69

--- ETAPA 3: REVISÃO (Revise) ---

O sistema sugeriu um preço de R$ 507,998.69.


Essa sugestão foi precisa? (s/n):  s


INFO: Solução confirmada pelo usuário.

--- ETAPA 4: RETENÇÃO (Retain) ---
Sucesso! Novo caso (ID: 53) foi aprendido e salvo na base.
INFO: Recalculando parâmetros do sistema com o novo caso...
Valores Min-Max para normalização calculados:


,min,max
metros_quadrados,42.0,160.0
num_quartos,1.0,4.0
num_banheiros,1.0,3.0
vagas_garagem,0.0,3.0


Pré-cálculo de adaptação concluído:
  -> Centro: Custo da Mobília = R$ 846.37 por m²
  -> Uvaranas: Custo da Mobília = R$ 889.07 por m²
  -> Jardim Carvalho: Custo da Mobília = R$ 1,141.17 por m²
  -> Oficinas: Custo da Mobília = R$ 451.85 por m²

--- Ciclo RBC Concluído ---
A base de casos agora possui 53 casos.


,metros_quadrados,num_quartos,num_banheiros,vagas_garagem,bairro,mobiliado,preco_vendido
51,50.0,1,1,1,Oficinas,False,234464.996969
52,80.0,2,1,1,Oficinas,True,401900.000000
53,80.0,2,1,1,Centro,True,507998.685060


## funcao auxiliar para remocao

In [10]:
# --- CÉLULA DE UTILITÁRIO: EXCLUIR ÚLTIMO CASO ---

print("Iniciando exclusão da última linha...")

# NOME_ARQUIVO_BASE_CASOS deve estar definido nas células anteriores
# (ex: "base_de_casos_50.csv")

# Verifica se a base de casos está carregada na memória
if 'df_base_de_casos' in globals() and not df_base_de_casos.empty:
    
    # Pega o ID da última linha (que será excluída) para reportar
    ultimo_id = df_base_de_casos.index[-1]
    
    print(f"A base atual tem {len(df_base_de_casos)} casos. O último caso (ID: {ultimo_id}) será excluído.")
    
    # 1. Remove a última linha do DataFrame EM MEMÓRIA
    # .iloc[:-1] seleciona todas as linhas, exceto a última
    df_base_de_casos = df_base_de_casos.iloc[:-1]
    
    # 2. Salva o DataFrame (encurtado) de volta no ARQUIVO CSV
    df_base_de_casos.to_csv(NOME_ARQUIVO_BASE_CASOS)
    
    print(f"Sucesso! Caso ID {ultimo_id} foi removido do arquivo {NOME_ARQUIVO_BASE_CASOS}.")
    
    # 3. Recalcula os parâmetros do sistema (MUITO IMPORTANTE!)
    # Isto atualiza as variáveis globais para o estado correto.
    print("Recalculando parâmetros do sistema (normalização e custos)...")
    ranges_normalizacao = calcular_ranges_normalizacao(df_base_de_casos)
    custos_mobilia_por_bairro = calcular_custos_por_bairro(df_base_de_casos)
    
    print(f"\nSistema pronto. A base de casos (em memória e em disco) agora possui {len(df_base_de_casos)} casos.")
    display(df_base_de_casos.tail()) # Mostra o novo final da base de dados
    
else:
    print("Erro: A base de dados ('df_base_de_casos') não está carregada na memória.")
    print("Por favor, execute a célula de 'Carregando a Base de Casos' primeiro.")

Iniciando exclusão da última linha...
A base atual tem 53 casos. O último caso (ID: 53) será excluído.
Sucesso! Caso ID 53 foi removido do arquivo base_de_casos.csv.
Recalculando parâmetros do sistema (normalização e custos)...
Valores Min-Max para normalização calculados:


,min,max
metros_quadrados,42.0,160.0
num_quartos,1.0,4.0
num_banheiros,1.0,3.0
vagas_garagem,0.0,3.0


Pré-cálculo de adaptação concluído:
  -> Centro: Custo da Mobília = R$ 849.98 por m²
  -> Uvaranas: Custo da Mobília = R$ 889.07 por m²
  -> Jardim Carvalho: Custo da Mobília = R$ 1,141.17 por m²
  -> Oficinas: Custo da Mobília = R$ 451.85 por m²

Sistema pronto. A base de casos (em memória e em disco) agora possui 52 casos.


,metros_quadrados,num_quartos,num_banheiros,vagas_garagem,bairro,mobiliado,preco_vendido
48,76.0,2,2,1,Centro,True,479900.000000
49,74.0,2,1,1,Uvaranas,False,305000.000000
50,112.0,3,2,2,Jardim Carvalho,True,810300.000000
51,50.0,1,1,1,Oficinas,False,234464.996969
52,80.0,2,1,1,Oficinas,True,401900.000000
